# Module 3 — Système d'Explicabilité (XAI)
**Mémoire M1 — Zinedine Hamadi & Sara Hadidi — Sorbonne Université 2025/2026**

## Plan
- **Étape 3.1** : Attention Visualization — identifier les tokens influents
- **Étape 3.2** : Templates linguistiques — générer des explications naturelles
- **Étape 3.3** : Intégration au pipeline complet
- **Étape 3.4** : Évaluation du système XAI

## Prérequis
- GPU T4 activé
- `emobert_v2` sur Drive
- `memoire_M1/pipeline_v1/strategies.yaml` sur Drive

## Cellule 0 — Installation & imports

In [1]:
!pip install -q transformers torch pyyaml matplotlib seaborn

from google.colab import drive
drive.mount('/content/drive')

import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import torch
import torch.nn.functional as F
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABELS    = ['stress', 'frustration', 'engagement', 'confusion', 'satisfaction', 'neutre']
THRESHOLD = 0.5
MODEL_PATH = '/content/drive/MyDrive/emobert_v2'

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('✓ Imports OK')

Mounted at /content/drive
GPU : Tesla T4
✓ Imports OK


## Étape 3.1 — Attention Visualization

In [2]:
# ── Charger le modèle avec output_attentions=True ─────────────────────────────
print('Chargement EmoBERT v2...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    output_attentions=True   # active la sortie des poids d'attention
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f'✓ Modèle chargé sur {device}')

Chargement EmoBERT v2...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Modèle chargé sur cuda


In [3]:
# ── Fonction d'extraction des tokens influents ────────────────────────────────
def get_attention_scores(text: str) -> dict:
    """
    Extrait les poids d'attention d'EmoBERT pour identifier
    les tokens qui ont le plus influencé la classification.

    Returns:
        dict avec label, confidence, tokens, scores d'attention
    """
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128, padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Prédiction de l'émotion
    probs      = F.softmax(outputs.logits, dim=-1).squeeze()
    confidence = probs.max().item()
    label_idx  = probs.argmax().item()
    label = LABELS[label_idx] if confidence >= THRESHOLD else 'neutre'

    # Extraction des poids d'attention
    # outputs.attentions = tuple de (n_layers,) tenseurs de shape (batch, heads, seq, seq)
    # On prend la dernière couche, moyenne sur toutes les têtes d'attention
    last_layer_attn = outputs.attentions[-1]  # dernière couche
    avg_attn = last_layer_attn.mean(dim=1).squeeze()  # moyenne sur les têtes

    # Score d'attention de chaque token = moyenne de l'attention reçue
    token_scores = avg_attn.mean(dim=0).cpu().numpy()

    # Récupérer les tokens décodés
    token_ids = inputs['input_ids'].squeeze().cpu().numpy()
    tokens    = tokenizer.convert_ids_to_tokens(token_ids)

    # Filtrer les tokens spéciaux (<s>, </s>, <pad>)
    filtered = [
        (tok, float(score))
        for tok, score in zip(tokens, token_scores)
        if tok not in ['<s>', '</s>', '<pad>']
    ]

    return {
        'label':      label,
        'confidence': round(confidence, 3),
        'all_scores': {l: round(p.item(), 3) for l, p in zip(LABELS, probs)},
        'tokens':     [t for t, _ in filtered],
        'attn_scores': [s for _, s in filtered]
    }


def get_top_tokens(result: dict, n: int = 3) -> list:
    """
    Retourne les n tokens les plus influents, nettoyés.
    """
    pairs = list(zip(result['tokens'], result['attn_scores']))
    pairs.sort(key=lambda x: x[1], reverse=True)

    top = []
    for tok, score in pairs[:n*2]:  # prendre plus pour filtrer
        # Nettoyer les préfixes RoBERTa (Ġ = espace)
        clean = tok.replace('Ġ', '').replace('▁', '').strip()
        if len(clean) > 2:  # ignorer les tokens très courts
            top.append((clean, round(score, 4)))
        if len(top) == n:
            break
    return top


# ── Test ──────────────────────────────────────────────────────────────────────
test_texts = [
    'Je suis trop stressé par le partiel de demain.',
    "J'ai encore faux, je comprends pas pourquoi ça marche pas.",
    "C'est fascinant, je veux vraiment comprendre ce concept !",
]

print('=== Test Attention Visualization ===')
for text in test_texts:
    result = get_attention_scores(text)
    top    = get_top_tokens(result, n=3)
    print(f'\nTexte     : {text}')
    print(f'Émotion   : {result["label"]} ({result["confidence"]:.0%})')
    print(f'Top tokens: {[(t, f"{s:.3f}") for t, s in top]}')

=== Test Attention Visualization ===

Texte     : Je suis trop stressé par le partiel de demain.
Émotion   : stress (52%)
Top tokens: [('stress', '0.147'), ('part', '0.085'), ('dem', '0.043')]

Texte     : J'ai encore faux, je comprends pas pourquoi ça marche pas.
Émotion   : engagement (73%)
Top tokens: [('rend', '0.046')]

Texte     : C'est fascinant, je veux vraiment comprendre ce concept !
Émotion   : engagement (80%)
Top tokens: [('est', '0.073'), ('fasc', '0.058')]


In [4]:
# ── Générer une heatmap d'attention ───────────────────────────────────────────
def plot_attention_heatmap(text: str, save_path: str = None):
    """
    Génère une heatmap des poids d'attention sur les tokens du texte.
    """
    result = get_attention_scores(text)
    tokens = result['tokens']
    scores = np.array(result['attn_scores'])

    # Normaliser entre 0 et 1
    scores_norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)

    fig, ax = plt.subplots(figsize=(max(8, len(tokens) * 0.6), 2.5))

    # Heatmap horizontale
    im = ax.imshow([scores_norm], cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(
        [t.replace('Ġ', '') for t in tokens],
        rotation=45, ha='right', fontsize=9
    )
    ax.set_yticks([])
    ax.set_title(
        f'Attention — {result["label"]} ({result["confidence"]:.0%})\n"{text[:60]}"',
        fontsize=10, pad=8
    )
    plt.colorbar(im, ax=ax, orientation='vertical', fraction=0.02, pad=0.04)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()


os.makedirs('xai_outputs', exist_ok=True)

# Générer les heatmaps pour les 3 textes de test
for i, text in enumerate(test_texts):
    plot_attention_heatmap(text, save_path=f'xai_outputs/heatmap_{i+1}.png')
    print(f'✓ heatmap_{i+1}.png sauvegardée')

print('\n✓ Étape 3.1 OK')

✓ heatmap_1.png sauvegardée
✓ heatmap_2.png sauvegardée
✓ heatmap_3.png sauvegardée

✓ Étape 3.1 OK


## Étape 3.2 — Templates linguistiques

In [5]:
# ── Créer xai_templates.yaml ──────────────────────────────────────────────────
xai_templates_content = """stress:
  template: >-
    J'ai détecté du stress dans ton message, notamment à travers les mots
    \"{top_tokens}\". Cela m'indique que tu ressens une pression ou une anxiété.
    C'est pourquoi j'adopte un ton rassurant et je décompose ma réponse
    en petites étapes pour t'aider à avancer sereinement.

frustration:
  template: >-
    J'ai détecté de la frustration dans ton message, notamment à travers les mots
    \"{top_tokens}\". Cela m'indique que tu te heurtes à un blocage répété.
    C'est pourquoi je commence par reconnaître ta difficulté et je reformule
    l'explication avec un exemple concret.

engagement:
  template: >-
    J'ai détecté de l'engagement dans ton message, notamment à travers les mots
    \"{top_tokens}\". Cela m'indique que tu es motivé et curieux.
    C'est pourquoi j'adopte un ton dynamique et je propose un approfondissement
    pour stimuler ta réflexion.

confusion:
  template: >-
    J'ai détecté de la confusion dans ton message, notamment à travers les mots
    \"{top_tokens}\". Cela m'indique que quelque chose n'est pas clair pour toi.
    C'est pourquoi je vérifie les prérequis et j'utilise des analogies simples.

satisfaction:
  template: >-
    J'ai détecté de la satisfaction dans ton message, notamment à travers les mots
    \"{top_tokens}\". Cela m'indique que tu as bien compris ou réussi.
    C'est pourquoi je renforce positivement et je fais le lien avec la
    prochaine notion.

neutre:
  template: >-
    Ton message ne présente pas d'émotion particulière détectée. J'adopte
    un ton pédagogique standard pour répondre de manière claire et structurée.
"""

with open('xai_templates.yaml', 'w', encoding='utf-8') as f:
    f.write(xai_templates_content)

with open('xai_templates.yaml', 'r', encoding='utf-8') as f:
    XAI_TEMPLATES = yaml.safe_load(f)

print('✓ xai_templates.yaml créé')


# ── Fonction de génération d'explication ─────────────────────────────────────
def generate_explanation(text: str) -> dict:
    """
    Génère une explication naturelle basée sur :
    - l'émotion détectée par EmoBERT
    - les tokens les plus influents (attention)
    - le template linguistique correspondant
    """
    # 1. Attention + prédiction
    result   = get_attention_scores(text)
    emotion  = result['label']
    top_toks = get_top_tokens(result, n=3)

    # 2. Formater les tokens pour le template
    top_tokens_str = ', '.join([f'"{t}"' for t, _ in top_toks])

    # 3. Remplir le template
    template = XAI_TEMPLATES[emotion]['template']
    explanation = template.replace('{top_tokens}', top_tokens_str)

    return {
        'emotion':       emotion,
        'confidence':    result['confidence'],
        'top_tokens':    top_toks,
        'explanation':   explanation
    }


# ── Tests : 6 cas (un par classe) ─────────────────────────────────────────────
test_cases = [
    'Je suis trop stressé par le partiel de demain.',
    "J'ai encore faux, je comprends pas pourquoi ça marche pas.",
    "C'est fascinant, je veux vraiment comprendre ce concept !",
    "Je vois pas du tout la différence entre les deux notions.",
    "Ah oui ! Maintenant c'est clair, j'ai enfin compris !",
    "Bonjour, pouvez-vous m'aider s'il vous plaît ?",
]

print('=== Tests Templates Linguistiques ===')
for text in test_cases:
    xai = generate_explanation(text)
    print(f'\nTexte      : {text}')
    print(f'Émotion    : {xai["emotion"]} ({xai["confidence"]:.0%})')
    print(f'Top tokens : {xai["top_tokens"]}')
    print(f'Explication: {xai["explanation"][:120]}...')

print('\n✓ Étape 3.2 OK')

✓ xai_templates.yaml créé
=== Tests Templates Linguistiques ===

Texte      : Je suis trop stressé par le partiel de demain.
Émotion    : stress (52%)
Top tokens : [('stress', 0.1472), ('part', 0.0845), ('dem', 0.043)]
Explication: J'ai détecté du stress dans ton message, notamment à travers les mots ""stress", "part", "dem"". Cela m'indique que tu r...

Texte      : J'ai encore faux, je comprends pas pourquoi ça marche pas.
Émotion    : engagement (73%)
Top tokens : [('rend', 0.0456)]
Explication: J'ai détecté de l'engagement dans ton message, notamment à travers les mots ""rend"". Cela m'indique que tu es motivé et...

Texte      : C'est fascinant, je veux vraiment comprendre ce concept !
Émotion    : engagement (80%)
Top tokens : [('est', 0.0726), ('fasc', 0.0582)]
Explication: J'ai détecté de l'engagement dans ton message, notamment à travers les mots ""est", "fasc"". Cela m'indique que tu es mo...

Texte      : Je vois pas du tout la différence entre les deux notions.
Émotion    :

## Étape 3.3 — Intégration au pipeline complet

In [6]:
# ── Charger les stratégies pédagogiques ───────────────────────────────────────
strategies_path = '/content/drive/MyDrive/memoire_M1/pipeline_v1/strategies.yaml'
with open(strategies_path, 'r', encoding='utf-8') as f:
    STRATEGIES = yaml.safe_load(f)

SYSTEM_BASE = """Tu es un tuteur pédagogique intelligent spécialisé en informatique
et sciences du langage. Ton rôle est d'aider les apprenants à comprendre des concepts
complexes de manière adaptée à leur état émotionnel.
Tu t'exprimes toujours en français, de manière claire et bienveillante.

{emotional_instruction}

Contexte émotionnel détecté : {emotion} (confiance : {confidence:.0%})
"""

def build_prompt(emotion: str, confidence: float) -> str:
    instruction = STRATEGIES.get(emotion, STRATEGIES['neutre'])['instruction']
    return SYSTEM_BASE.format(
        emotional_instruction=instruction,
        emotion=emotion,
        confidence=confidence
    )

print('✓ Stratégies chargées')

✓ Stratégies chargées


In [7]:
# ── Charger le LLM ────────────────────────────────────────────────────────────
from transformers import pipeline as hf_pipeline

print('Chargement Zephyr-7B...')
pipe = hf_pipeline(
    'text-generation',
    model='HuggingFaceH4/zephyr-7b-beta',
    torch_dtype=torch.float16,
    device_map='auto',
)
print('✓ LLM chargé')

def generate(system_prompt: str, user_message: str, history: list = []) -> str:
    messages = [{'role': 'system', 'content': system_prompt}]
    messages += history
    messages.append({'role': 'user', 'content': user_message})
    output = pipe(messages, max_new_tokens=300)
    return output[0]['generated_text'][-1]['content'].strip()

print('✓ generate() prête')

Chargement Zephyr-7B...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

✓ LLM chargé
✓ generate() prête


In [8]:
# ── EmotionalTutor v2 avec XAI ────────────────────────────────────────────────
import datetime

os.makedirs('logs', exist_ok=True)

class DialogueHistory:
    def __init__(self, max_turns=5):
        self.history = []; self.max_turns = max_turns
    def add(self, role, content):
        self.history.append({'role': role, 'content': content})
        if len(self.history) > self.max_turns * 2:
            self.history = self.history[-self.max_turns * 2:]
    def get(self): return self.history.copy()
    def reset(self): self.history = []


class EmotionalTutorXAI:
    """
    Pipeline complet avec module XAI :
    EmoBERT → Attention → Template → PromptBuilder → LLM → Logger
    """
    def __init__(self):
        self.history = DialogueHistory(max_turns=5)
        self.logs    = []
        print('✓ EmotionalTutorXAI initialisé')

    def respond(self, user_message: str) -> dict:
        # 1. Détection émotion + attention
        xai_result  = generate_explanation(user_message)
        emotion     = xai_result['emotion']
        confidence  = xai_result['confidence']
        explanation = xai_result['explanation']
        top_tokens  = xai_result['top_tokens']

        # 2. Construction du prompt adaptatif
        system_prompt = build_prompt(emotion, confidence)

        # 3. Génération réponse LLM
        response = generate(system_prompt, user_message, self.history.get())

        # 4. Mise à jour historique
        self.history.add('user',      user_message)
        self.history.add('assistant', response)

        # 5. Logging enrichi avec XAI
        log_entry = {
            'timestamp':    datetime.datetime.now().isoformat(),
            'user_message': user_message,
            'emotion':      emotion,
            'confidence':   confidence,
            'top_tokens':   top_tokens,
            'explanation':  explanation,
            'llm_response': response
        }
        self.logs.append(log_entry)
        with open('logs/dialogues_xai.jsonl', 'a', encoding='utf-8') as f:
            f.write(json.dumps(log_entry, ensure_ascii=False) + '\n')

        return {
            'response':    response,
            'emotion':     emotion,
            'confidence':  confidence,
            'explanation': explanation,
            'top_tokens':  top_tokens
        }

    def reset(self):
        self.history.reset()


tutor_xai = EmotionalTutorXAI()
print('\n✓ Étape 3.3 OK')

✓ EmotionalTutorXAI initialisé

✓ Étape 3.3 OK


## Étape 3.4 — Démonstration avec XAI

In [9]:
tutor_xai.reset()

def demo_turn_xai(message: str):
    result = tutor_xai.respond(message)
    print(f'Apprenant   : {message}')
    print(f'Émotion     : {result["emotion"]} ({result["confidence"]:.0%})')
    print(f'Top tokens  : {result["top_tokens"]}')
    print(f'Explication : {result["explanation"][:150]}...')
    print(f'Tuteur      : {result["response"][:200]}...')
    print('-' * 70)

print('=== DÉMONSTRATION — Pipeline avec XAI ===')
print()

demo_turn_xai('Je ne comprends pas du tout ce qu\'est un arbre binaire.')
demo_turn_xai('J\'ai essayé 5 fois et mon code plante encore.')
demo_turn_xai('Super ! J\'ai réussi l\'insertion !')
demo_turn_xai('C\'est fascinant, je veux comprendre les arbres AVL maintenant.')
demo_turn_xai('Je suis trop stressé par le partiel de demain.')

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


=== DÉMONSTRATION — Pipeline avec XAI ===



[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant   : Je ne comprends pas du tout ce qu'est un arbre binaire.
Émotion     : neutre (34%)
Top tokens  : [('comp', 0.0489)]
Explication : Ton message ne présente pas d'émotion particulière détectée. J'adopte un ton pédagogique standard pour répondre de manière claire et structurée....
Tuteur      : Un arbre binaire, aussi appelé arbre de recherche, est un type d'arbre qui permet de stocker et de rechercher des données de manière rapide et efficace, en particulier dans le cadre de grands ensemble...
----------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant   : J'ai essayé 5 fois et mon code plante encore.
Émotion     : engagement (77%)
Top tokens  : [('ois', 0.0683), ('mon', 0.0642)]
Explication : J'ai détecté de l'engagement dans ton message, notamment à travers les mots ""ois", "mon"". Cela m'indique que tu es motivé et curieux. C'est pourquoi...
Tuteur      : Bien que la compréhension théorique de la structure d'un arbre binaire soit importante, l'implémentation pratique en code peut être complexe à débuter. Voici quelques idées pour vous aider à débugger ...
----------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant   : Super ! J'ai réussi l'insertion !
Émotion     : neutre (47%)
Top tokens  : [('insert', 0.1101), ('Super', 0.0889), ('ion', 0.0797)]
Explication : Ton message ne présente pas d'émotion particulière détectée. J'adopte un ton pédagogique standard pour répondre de manière claire et structurée....
Tuteur      : C'est merveilleux que vous ayez réussi à insérer une valeur dans l'arbre binaire ! Continuez ainsi avec confiance et continuez à utiliser les techniques de débugage que j'ai suggestées précédemment lo...
----------------------------------------------------------------------


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Apprenant   : C'est fascinant, je veux comprendre les arbres AVL maintenant.
Émotion     : engagement (76%)
Top tokens  : [('les', 0.0527), ('est', 0.0472), ('main', 0.0455)]
Explication : J'ai détecté de l'engagement dans ton message, notamment à travers les mots ""les", "est", "main"". Cela m'indique que tu es motivé et curieux. C'est ...
Tuteur      : Les arbres AVL sont une variété d'arbres binaires auto-équilibrés qui garantissent une hauteur constante de l'arbre, ce qui signifie que les arbres sont toujours hauts de deux à trois niveaux, même lo...
----------------------------------------------------------------------
Apprenant   : Je suis trop stressé par le partiel de demain.
Émotion     : stress (52%)
Top tokens  : [('stress', 0.1472), ('part', 0.0845), ('dem', 0.043)]
Explication : J'ai détecté du stress dans ton message, notamment à travers les mots ""stress", "part", "dem"". Cela m'indique que tu ressens une pression ou une anx...
Tuteur      : Je suis désolé de savoir que 

In [10]:
# ── Analyse des logs XAI ──────────────────────────────────────────────────────
import pandas as pd

logs = [json.loads(l) for l in open('logs/dialogues_xai.jsonl', encoding='utf-8')]
df   = pd.DataFrame(logs)[['user_message', 'emotion', 'confidence', 'top_tokens']]
df['confidence'] = df['confidence'].apply(lambda x: f'{x:.0%}')
df['top_tokens'] = df['top_tokens'].apply(lambda x: ', '.join([t for t, _ in x[:2]]))

print('=== Logs XAI ===')
print(df.to_string(index=False))
print(f'\n✓ {len(logs)} tours loggés avec explications')

=== Logs XAI ===
                                                  user_message    emotion confidence    top_tokens
       Je ne comprends pas du tout ce qu'est un arbre binaire.     neutre        34%          comp
                 J'ai essayé 5 fois et mon code plante encore. engagement        77%      ois, mon
                             Super ! J'ai réussi l'insertion !     neutre        47% insert, Super
C'est fascinant, je veux comprendre les arbres AVL maintenant. engagement        76%      les, est
                Je suis trop stressé par le partiel de demain.     stress        52%  stress, part

✓ 5 tours loggés avec explications


In [11]:
# ── Sauvegarder tout sur Drive ────────────────────────────────────────────────
import shutil

drive_xai = '/content/drive/MyDrive/memoire_M1/xai_outputs'
os.makedirs(drive_xai, exist_ok=True)

# Heatmaps
for f in os.listdir('xai_outputs'):
    shutil.copy(f'xai_outputs/{f}', f'{drive_xai}/{f}')

# Templates
shutil.copy('xai_templates.yaml', f'{drive_xai}/xai_templates.yaml')

# Logs
shutil.copy('logs/dialogues_xai.jsonl', f'{drive_xai}/dialogues_xai.jsonl')

# Fichier Python du module XAI
xai_module_code = '''
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F
import yaml

LABELS    = ["stress", "frustration", "engagement", "confusion", "satisfaction", "neutre"]
THRESHOLD = 0.5

class XAIExplainer:
    def __init__(self, model_path, templates_path="xai_templates.yaml"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model     = AutoModelForSequenceClassification.from_pretrained(
            model_path, output_attentions=True)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()
        with open(templates_path, "r", encoding="utf-8") as f:
            self.templates = yaml.safe_load(f)

    def explain(self, text: str) -> dict:
        inputs = self.tokenizer(text, return_tensors="pt",
            truncation=True, max_length=128, padding=True).to(self.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        probs      = F.softmax(outputs.logits, dim=-1).squeeze()
        confidence = probs.max().item()
        label      = LABELS[probs.argmax().item()] if confidence >= THRESHOLD else "neutre"
        attn       = outputs.attentions[-1].mean(dim=1).squeeze().mean(dim=0).cpu().numpy()
        token_ids  = inputs["input_ids"].squeeze().cpu().numpy()
        tokens     = self.tokenizer.convert_ids_to_tokens(token_ids)
        filtered   = [(t.replace("Ġ","").strip(), float(s))
                      for t, s in zip(tokens, attn)
                      if t not in ["<s>","</s>","<pad>"] and len(t.replace("Ġ","")) > 2]
        top        = sorted(filtered, key=lambda x: x[1], reverse=True)[:3]
        top_str    = ", ".join([f\'"{t}"\' for t, _ in top])
        template   = self.templates[label]["template"]
        explanation = template.replace("{top_tokens}", top_str)
        return {"label": label, "confidence": round(confidence, 3),
                "top_tokens": top, "explanation": explanation}
'''

with open(f'{drive_xai}/xai_explainer.py', 'w', encoding='utf-8') as f:
    f.write(xai_module_code.strip())

print(f'✓ Tous les fichiers XAI sauvegardés dans {drive_xai}')
print('  - heatmap_1.png, heatmap_2.png, heatmap_3.png')
print('  - xai_templates.yaml')
print('  - xai_explainer.py')
print('  - dialogues_xai.jsonl')
print('\n✓ Module 3 XAI complet !')

✓ Tous les fichiers XAI sauvegardés dans /content/drive/MyDrive/memoire_M1/xai_outputs
  - heatmap_1.png, heatmap_2.png, heatmap_3.png
  - xai_templates.yaml
  - xai_explainer.py
  - dialogues_xai.jsonl

✓ Module 3 XAI complet !
